In [1]:
import warnings, math
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

import torch
import torch.nn as nn

from transformers import (AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig, TrainingArguments, Trainer,)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from torch.utils.data import Dataset

import transformers, accelerate, peft
print("torch        :", torch.__version__)
print("transformers :", transformers.__version__)
print("accelerate   :", accelerate.__version__)
print("peft         :", peft.__version__)
print("CUDA         :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU          :", torch.cuda.get_device_name(0))
    print("VRAM (GB)    :", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
    print("bf16 support :", torch.cuda.is_bf16_supported())

torch        : 2.12.1+cu130
transformers : 5.12.1
accelerate   : 1.14.0
peft         : 0.19.1
CUDA         : True
GPU          : NVIDIA RTX A4000
VRAM (GB)    : 16.7
bf16 support : True


In [ ]:
## Configuration
SEED        = 42
MODEL_NAME  = "Qwen/Qwen3-1.7B"
MAX_LEN     = 128

BATCH_SIZE            = 8
GRAD_ACCUM_STEPS      = 2
NUM_EPOCHS            = 3           
LR                    = 2e-4        
WARMUP_RATIO          = 0.1

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = False        # keep compute in bf16, consistent with the DeBERTa run
print(f"Compute dtype: {'bf16' if USE_BF16 else 'fp32'}")

ANNOTATED_FILE = Path("/user/HS402/kk01697/Documents/dissertation/story-evaluation-dissertation/data/processed/annotation_sample_annotated.csv")
OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
MODEL_DIR = OUTPUT_DIR / "qwen3_lora_classifier"

CUSTOM_DIM = ["Narrative Structure & Quality","Character & Emotion", "Originality", "Immersion","Thematic Depth", "Writing Style",]

Compute dtype: bf16


In [4]:
ann_df = pd.read_csv(ANNOTATED_FILE)
ann_df = ann_df.dropna(subset=["sentence"])

print("Shape:", ann_df.shape)
ann_df.head()

Shape: (3000, 10)


,review_id,sentence_idx,sentence,language,Narrative Structure & Quality,Character & Emotion,Originality,Immersion,Thematic Depth,Writing Style
0,6c98fe733ae0c27671ebbbe68b77fd8f,6,The initial deepening of the mechanics of the ...,eng,0,0,0,1,0,0
1,cd8abbbf2727515f904a6b189cb0eb84,24,I think that's more troubling when it comes to...,eng,0,0,0,0,0,0
2,b0d8887563f48d59440cd78144ef23c0,59,The one note simple tone of everything leads m...,en-US,0,0,0,0,0,1
3,93060ddc1ef84b111915ed91cfc443de,35,Saving grace was that he was the only characte...,eng,0,1,0,0,0,0
4,4fac8a39591a791eb0a78c4c08176d2b,4,I've never been so disappointed by this author.,eng,0,0,0,0,0,0


In [5]:
class SentenceDataset(Dataset):
    def __init__(self, texts: list[str], labels: np.ndarray, tokenizer):
        self.enc = tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids":      self.enc["input_ids"][idx],
            "attention_mask": self.enc["attention_mask"][idx],
            "labels":         self.labels[idx],
        }

In [6]:
class NaNGuardTrainer(Trainer):
    def __init__(self, *args, pos_weight: torch.Tensor = None, **kwargs):
        super().__init__(*args, **kwargs)
        self._pos_weight = pos_weight.to(DEVICE) if pos_weight is not None else None
        self._nan_batches = 0

    # ── custom loss ───────────────────────────────────────────────────────────
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits.float()      # force fp32 even in bf16 runs
        labels  = labels.float()

        loss_fn = nn.BCEWithLogitsLoss(pos_weight=self._pos_weight)
        loss    = loss_fn(logits, labels)

        # detect and report NaN loss without crashing
        if torch.isnan(loss):
            self._nan_batches += 1
            print(f"[NaNGuard] NaN loss detected (batch #{self._nan_batches}) — skipping")
            loss = torch.tensor(0.0, requires_grad=True, device=DEVICE)

        return (loss, outputs) if return_outputs else loss

    def training_step(self, model, inputs, num_items_in_batch=None):
        loss = super().training_step(model, inputs, num_items_in_batch)

        # zero any NaN/Inf gradients before the optimizer touches them
        # (with LoRA this only scans the small set of trainable adapter params)
        nan_params = []
        for name, param in model.named_parameters():
            if param.requires_grad and param.grad is not None:
                bad = torch.isnan(param.grad) | torch.isinf(param.grad)
                if bad.any():
                    param.grad[bad] = 0.0
                    nan_params.append(name)
        if nan_params:
            print(f"[NaNGuard] Zeroed NaN/Inf grads in: {nan_params[:3]}{'...' if len(nan_params)>3 else ''}")

        return loss

In [7]:
def build_compute_metrics(threshold: float = 0.5):
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        probs = 1.0 / (1.0 + np.exp(-logits))    # sigmoid
        preds = (probs >= threshold).astype(int)

        micro = f1_score(labels, preds, average="micro", zero_division=0)
        macro = f1_score(labels, preds, average="macro", zero_division=0)
        per   = f1_score(labels, preds, average=None,    zero_division=0)

        metrics = {"f1_micro": micro, "f1_macro": macro}
        for dim, score in zip(CUSTOM_DIM, per):
            metrics[f"f1_{dim}"] = score
        return metrics

    return compute_metrics

In [8]:
torch.manual_seed(SEED)
np.random.seed(SEED)

sentences = ann_df["sentence"].tolist()
labels    = ann_df[CUSTOM_DIM].values.astype(np.float32)

idx = np.arange(len(sentences))
idx_trainval, idx_test = train_test_split(idx, test_size=0.15, random_state=SEED)
idx_train, idx_val     = train_test_split(idx_trainval, test_size=0.15, random_state=SEED)

train_labels = labels[idx_train]
pos_counts   = train_labels.sum(axis=0).clip(min=1)
neg_counts   = len(train_labels) - pos_counts
pos_weight   = torch.tensor(neg_counts / pos_counts, dtype=torch.float32)

print(f"Split — train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}\n")
print(f"{'Dimension':<35} {'Pos':>5}  {'pos_weight':>10}")
print("-" * 55)
for dim, p, w in zip(CUSTOM_DIM, pos_counts.astype(int), pos_weight.tolist()):
    print(f"{dim:<35} {p:>5}  {w:>10.1f}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

train_ds = SentenceDataset([sentences[i] for i in idx_train], train_labels,      tokenizer)
val_ds   = SentenceDataset([sentences[i] for i in idx_val],   labels[idx_val],   tokenizer)
test_ds  = SentenceDataset([sentences[i] for i in idx_test],  labels[idx_test],  tokenizer)

print("Datasets created.")
print("Pad token:", tokenizer.pad_token, "| id:", tokenizer.pad_token_id)
print("Sample labels:", train_ds[0]["labels"])

Split — train: 2167  val: 383  test: 450

Dimension                             Pos  pos_weight
-------------------------------------------------------
Narrative Structure & Quality         335         5.5
Character & Emotion                   417         4.2
Originality                            80        26.1
Immersion                              66        31.8
Thematic Depth                         54        39.1
Writing Style                         125        16.3


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Datasets created.
Pad token: <|endoftext|> | id: 151643
Sample labels: tensor([0., 0., 0., 0., 0., 0.])


In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(CUSTOM_DIM),
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    device_map="auto",
)
model.config.pad_token_id = tokenizer.pad_token_id
model.gradient_checkpointing_enable()
model.enable_input_require_grads() 

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    # the classification head is randomly initialised and small; train it in full
    modules_to_save=["score"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print("Model on   :", next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/25.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Could not set the permissions on the file '/user/HS402/kk01697/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/blobs/912becff8d60672aa8628ef08c05898d9adf17c2ad4ae3caf99b065622fdeff9.d764142d.incomplete'. Error: [Errno 28] No space left on device: '/user/HS402/kk01697/.cache/huggingface/hub/models--Qwen--Qwen3-1.7B/blobs/tmp_7cedd790-d701-4d6a-a5f9-fac29cce5fd6'.
Continuing without setting permissions.


OSError: [Errno 28] No space left on device

In [ ]:
_sample = {k: v.unsqueeze(0).to(DEVICE) for k, v in train_ds[0].items() if k != "labels"}
_label  = train_ds[0]["labels"].unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    _out = model(**_sample)
model.train()

_loss_fn    = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(DEVICE))
_check_loss = _loss_fn(_out.logits.float(), _label.float())

print(f"Logits : {_out.logits.float().cpu()}")
print(f"Loss   : {_check_loss.item():.4f}")

assert not torch.isnan(_check_loss), "NaN loss before training starts — check data!"
assert _check_loss.item() < 20,      "Loss is unreasonably large — check pos_weight!"
print("\nSanity check passed \u2713")